# Deliverable 2

In [10]:
# Imports

import re
import json
import pandas as pd
from pathlib import Path
from charset_normalizer import from_path
import numpy as np

#### Task 1

In [11]:
test_path = Path("data/2026/student_union/POS_01/register_sales.log")

def file_reader(file_path: Path) -> tuple[str, str]:
    """
    Reads a file checking its encoding.

    args:
        file_path: The path for the file to be checked
    
    returns:
        text: the encoded text
        encoding: the encoding of the given text

    """
    raw_bytes = Path(file_path).read_bytes()

    # Checking for utf-8 encoding with BOM
    if raw_bytes[:3] == b"\xef\xbb\xbf":
        encoding = "utf-8-sig"

    # Otherwise try to guess correct encoding
    else:
        best_match = from_path(file_path).best()
        if best_match is None:
            raise ValueError(f"Could not detect encoding of {file_path}")
        encoding = best_match.encoding

    decoded_text = raw_bytes.decode(encoding)
    return decoded_text, encoding

text, encoding = file_reader(test_path)
#print(f"Encoding: {encoding}\n{text}")

For de log_parser() I've used lecture 6 for regex syntax

In [12]:
def log_parser(decoded_text: str) -> list[dict][str, str | float]:
    """
    Uses regular expression to capture 6 key fields from valid lines in a log file and
    ignores noise lines.

    Key fields:
        timestamp
        register_id
        item
        price
        currency
        status
    
    args:
        decoded_text: a string of from decoded file.

    returns: 
        list: List of dictonaries with valid transaction record
    """

    verbose_pattern = re.compile(
        r"""                                   
        \[(?P<timestamp> \d{4}-\d{2}-\d{2}\ \d{2}:\d{2}:\d{2})\]    # datetime YYYY-MM-DD HH:MM:SS
        \s+                                                         # whitespace
        \[(?P<register_id> POS_\d{2})\]                             # POS_xx
        \s+                                                         # whitespace
        item=(?P<item>\S+)                                          # item
        \s+                                                         # whitespace
        price=(?P<price>\S+)                                        # price
        \s+                                                         # whitespace
        (?P<currency>[A-Za-z]{3})                                   # currency
        \s+                                                         # whitespace
        \( (?P<status>[\w-]+) \)                                    # status
        $                                   
    """,
        re.VERBOSE,
    )    

    valid_records = []
    for line in decoded_text.splitlines():
        m = verbose_pattern.match(line)

        # check for non-valid lines
        if not m:
            continue

        valid_records.append(
            {
            "timestamp": m.group("timestamp"),
            "register_id": m.group("register_id"),
            "item": m.group("item"),
            "price": float(m.group("price")), # converts str to float here
            "currency": m.group("currency"),
            "status": m.group("status"),
            }
        )

    return valid_records

log_parser(text)

[{'timestamp': '2026-10-15 07:00:32',
  'register_id': 'POS_01',
  'item': 'Latte',
  'price': 4.3,
  'currency': 'USD',
  'status': 'REFUND'},
 {'timestamp': '2026-10-15 07:02:08',
  'register_id': 'POS_01',
  'item': 'Espresso',
  'price': 3.19,
  'currency': 'USD',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:04:05',
  'register_id': 'POS_01',
  'item': 'Sandwich',
  'price': 7.5,
  'currency': 'EUR',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:04:32',
  'register_id': 'POS_01',
  'item': 'Cappuccino',
  'price': 4.45,
  'currency': 'NOK',
  'status': 'CANCELLED'},
 {'timestamp': '2026-10-15 07:07:32',
  'register_id': 'POS_01',
  'item': 'Espresso',
  'price': 2.69,
  'currency': 'NOK',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:09:38',
  'register_id': 'POS_01',
  'item': 'Sandwich',
  'price': 7.72,
  'currency': 'NOK',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:11:45',
  'register_id': 'POS_01',
  'item': 'Croissant',
  'price': 4.26,
  '

#### Task 2

In [13]:
# test path
test_path = ("data/2026/library_cafe")

def recursive_dir_search(input_path: Path | str) -> list[Path]:
    """
    Recursively search through a directory hierarchy finding .log files.

    arg:
        input_path: This can be a path object or a string for the location for where
        to start the search.

    return:
        path_list: A list with the path to the found .log files.
    """
    inp_pat = Path(input_path)

    path_list = []
    for i in inp_pat.iterdir():
        if i.is_dir():
            path_list.extend(recursive_dir_search(i))
        elif i.is_file() and i.suffix == ".log":
            path_list.append(i)

    return path_list

log_files = recursive_dir_search(test_path)
print(log_files)

[WindowsPath('data/2026/library_cafe/POS_01/register_sales.log'), WindowsPath('data/2026/library_cafe/POS_02/register_sales.log'), WindowsPath('data/2026/library_cafe/POS_03/register_sales.log')]


In [14]:
test_path_json = ("data/store_config.json")

def traverse_nested(data: dict | list) -> set[str]:

    """
    Goes through a nested file of dicts and lists,
    recursivly checking for strings where "active": true.

    arg:
        data: parsed json data

    return:
        active_id: a set of register id strings where "active": true

    """
    active_id = set()

    if isinstance(data, dict):
        if data.get("active") is True and isinstance(data.get("id"), str):
            active_id.add(data["id"])

        for value in data.values():
            active_id.update(traverse_nested(value))

    elif isinstance(data, list):
        for item in data:
            active_id.update(traverse_nested(item))

    return active_id

with open(test_path_json) as f:
    config = json.load(f)

traverse_nested(config)

{'POS_01', 'POS_02', 'POS_03'}

#### Task 3

In [15]:
# Task 3.1 Remove Price Outliers
def remove_price_outliers(prices: list[float] | np.ndarray) -> np.ndarray:
    """Converts input price data to a 1D Numpy float array, detects hardware glitch

    outliers (|x - mu| > 3 * sigma) and replaces them with the median of valid prices.

    args:
        prices: List or NmuPy array of transaction prices values.

    returns:
        cleaned_prices: 1D NumPy float array with outliers replaces by valid median value.
    """

    # 1. Convert input data to a 1D NumPy float array
    price_arr = np.asanyarray(prices, dtype=float).ravel()

    if price_arr.size == 0:
        return price_arr


    # 2. Calculate mean (mu) and standard deviation (sigma)
    mean = np.mean(price_arr)
    std = np.std(price_arr)


    # 3. Detect outliers (|x - mean| > 3 * std)
    outliers_mask = np.abs(price_arr - mean) > (3 * std)


    # 4. Calculate the median of valid (non-outlier) prices
    valid_prices = price_arr[~outliers_mask]
    if len(valid_prices) > 0:
        valid_median = np.median(valid_prices)
    else:
        valid_median = np.median(price_arr)

    # 5. Replace outliers with the valid median and return cleaned array
    cleaned_prices = price_arr.copy()
    cleaned_prices[outliers_mask] = valid_median

    return cleaned_prices

# Task 3.2 Compute Refund Ratio
def compute_refund_ratio(
        sales_count: np.ndarray, refund_count: np.ndarray
    )    -> np.ndarray:
    """Computes the element wise refund ratio using Numpy array operations:

    Refund Ratio = Refund Count / (Sales Count + 10^-6)

    args:
        sales_count: Numpy array containing sales counts per item.
        refund_count: Numpy array containing refund counts per item.
    
    returns:
        refund_ratio: Numpy array of calculated refund ratios.
    """

    sales = np.asarray(sales_count, dtype=float)
    refunds = np.asarray(refund_count, dtype=float)

    epsilon = 1e-6

    return refunds / (sales + epsilon)



In [16]:
# Test task 3
test_prices = [
    3.50,
    4.20,
    3.80,
    4.00,
    3.60,
    4.10,
    3.90,
    4.00,
    3.70,
    4.20,
    3.80,
    4.10,
    3.90,
    4.00,
    3.50,
    4.20,
    999.00,
    3.80,
    3.90,
    4.10,
]
cleaned = remove_price_outliers(test_prices)

print("--- Test 3.1: remove_price_outliers ---")
print("Originale priser:", test_prices)
print("Vaskede priser:  ", cleaned)

# Test Task 3.2: Beregne refusjonsrate
sales = np.array([100, 50, 10])
refunds = np.array([5, 10, 0])
ratios = compute_refund_ratio(sales, refunds)

print("\n--- Test 3.2: compute_refund_ratio ---")
print("Sales:        ", sales)
print("Refunds:      ", refunds)
print("Refund ratios:", ratios)

--- Test 3.1: remove_price_outliers ---
Originale priser: [3.5, 4.2, 3.8, 4.0, 3.6, 4.1, 3.9, 4.0, 3.7, 4.2, 3.8, 4.1, 3.9, 4.0, 3.5, 4.2, 999.0, 3.8, 3.9, 4.1]
Vaskede priser:   [3.5 4.2 3.8 4.  3.6 4.1 3.9 4.  3.7 4.2 3.8 4.1 3.9 4.  3.5 4.2 3.9 3.8
 3.9 4.1]

--- Test 3.2: compute_refund_ratio ---
Sales:         [100  50  10]
Refunds:       [ 5 10  0]
Refund ratios: [0.05 0.2  0.  ]


#### Task 4

In [ ]:
# Task 4.1: Create DataFrame
def create_dataframe(transactions: list[dict]) -> pd.DataFrame:
    """
    Creates a Pandas DataFrame from a list of transaction dictionaries.

    args:
        transactions: List of dictionaries containing transaction data.

    returns:
        df: DataFrame with a DatetimeIndex named timestamp, sorted chronologically.
    """

    df = pd.DataFrame(transactions)

    df["timestamp"] = pd.to_datetime(df["timestamp"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
    
    df.set_index("timestamp", inplace=True)
    df.sort_index(inplace=True)

    return df


In [ ]:
# Task 4.2: Hourly revenue

def hourly_revenue(df: pd.DataFrame) -> pd.DataFrame:
    """ 
    Calculates the hourly revenue for each item in the DataFrame.

    args:
    df: Transaction DataFrame with a DatetimeIndex and columns item and price.

    returns:
        hourly: DataFrame with an (item, timestamp) MultiIndex and a price
            column containing the summed prices per item per hour.
        
    """

    grouped = df.groupby("item")
    hourly = grouped[["price"]].resample("1h").sum()

    return hourly


#### Task 5